# Options from Zero · Episode 1: Options are insurance

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate. All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. The running trade · 3. A call's payoff · 4. Payoff is not profit · 5. The put · 6. Buyer and seller · 7. A straddle · 8. Check against QuantLib · 9. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 1,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The option formulas used throughout the series (checked against QuantLib below):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. The running trade

A **call** gives its buyer the right, but not the obligation, to buy the share at the **strike** price up to **expiry**; a **put** gives the right to sell. The buyer pays the **premium** up front (ASX, *Options course, Module 2*).

The two premiums below come from the Black-Scholes formula. We use them as given for now: where they come from is what the rest of Part 1 is about (Episode 10 derives the call's price step by step).

In [ ]:
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20
call_premium = black_scholes(S, K, T, r, vol, call=True)
put_premium = black_scholes(S, K, T, r, vol, call=False)
out["trade"] = {"spot": S, "strike": K, "years": T, "rate_pct": round(r * 100, 6), "vol_pct": round(vol * 100, 6),
                "call": call_premium, "put": put_premium,
                "call_breakeven": K + call_premium, "put_breakeven": K - put_premium}
out["trade"]

## 3. A call's payoff

At expiry the call is worth $\max(S_T - K, 0)$: nothing if the share ends below the strike (you would not use the right), and one dollar more for every dollar above it.

In [ ]:
grid = np.arange(60.0, 150.5, 1.0)          # share price at expiry; includes the strike and every whole dollar
ticks = [{"x": float(x), "label": f"${x:.0f}"} for x in range(60, 151, 15)]
table_prices = [80.0, 90.0, 100.0, 105.0, 110.0, 120.0, 130.0]
call_pay = call_payoff(grid, K)
put_pay = put_payoff(grid, K)
out["chart"] = {"x": grid.tolist(), "ticks": ticks, "strike": [K],
                "call_payoff": call_pay.tolist(), "call_profit": (call_pay - call_premium).tolist(),
                "put_payoff": put_pay.tolist(), "put_profit": (put_pay - put_premium).tolist()}
out["call_points"] = {"x": [80.0, 110.0, 120.0, 130.0], "y": call_payoff([80, 110, 120, 130], K).tolist()}
out["call_at"] = {f"s{int(x)}": float(call_payoff(x, K)) for x in (80, 110, 120, 130)}
pd.DataFrame({"share at expiry": table_prices, "call payoff": call_payoff(table_prices, K)})

## 4. Payoff is not profit

The buyer paid the premium at the start, so profit at expiry is payoff minus premium. (Strictly, the premium was paid a year earlier and could have earned interest; on a payoff diagram it is conventional to subtract it as it is.) The **break-even** share price is where profit is zero: strike plus premium.

In [ ]:
rows = [{"share": x, "call_payoff": float(call_payoff(x, K)), "call_profit": float(call_payoff(x, K) - call_premium),
         "put_payoff": float(put_payoff(x, K)), "put_profit": float(put_payoff(x, K) - put_premium)}
        for x in table_prices]
out["table"] = {"rows": rows}
assert all(abs(r_["call_payoff"] - call_premium - r_["call_profit"]) < 1e-12 for r_ in rows)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for a, name in zip(ax, ["call", "put"]):
    a.plot(grid, out["chart"][f"{name}_payoff"], label="payoff")
    a.plot(grid, out["chart"][f"{name}_profit"], label="profit", ls="--")
    a.axhline(0, color="k", lw=.6); a.axvline(K, color="r", lw=.6)
    a.set(title=f"{name}, strike {K:.0f}", xlabel="share price at expiry"); a.legend(); a.grid(alpha=.3)
pd.DataFrame(rows)

## 5. The put

A put pays $\max(K - S_T, 0)$: it pays when the share falls, which is why it works like insurance on the share. Below the strike, every dollar the share loses, the put gains.

Why does the put cost more than the call here? The strike, \$105, is above today's price; Episode 4 shows exactly how the strike, the interest rate and the two premiums are tied together (put-call parity).

In [ ]:
# A share owner who buys the put: the worst case at expiry is the strike, whatever happens to the share.
floor = [float(x + put_payoff(x, K)) for x in (50.0, 80.0, 105.0, 130.0)]
out["put"] = {"premium": put_premium, "breakeven": K - put_premium, "insured_floor": K,
              "share_plus_put_at": floor, "put_dearer_than_call": put_premium > call_premium}
out["put"]

## 6. Buyer and seller

Every option has a buyer (ASX: the *taker*) and a seller (the *writer*). The seller receives the premium and takes on the obligation. Whatever the buyer makes, the seller loses, and the other way round: their profits add up to zero. The buyer can lose at most the premium; the writer of a call faces potentially unlimited losses (ASX, *Understanding Options Trading*, pages 3 and 25).

In [ ]:
buyer = call_payoff(grid, K) - call_premium
seller = call_premium - call_payoff(grid, K)
assert np.allclose(buyer + seller, 0)
out["sides"] = {"buyer_max_loss": call_premium, "seller_max_gain": call_premium,
                "seller_profit_at_130": float(call_premium - call_payoff(130.0, K)),   # the end-card question
                "seller_loss_at_150": float(call_payoff(150.0, K) - call_premium)}
out["sides"]

## 7. A straddle

Buy a call and a put with the same strike: you gain if the share moves a lot, in either direction. We use a strike of \$100 (today's price). This is the first appearance of **volatility**: the straddle is a bet on how much the share moves, not which way.

In [ ]:
Ks = 100.0
c100, p100 = black_scholes(S, Ks, T, r, vol, True), black_scholes(S, Ks, T, r, vol, False)
cost = c100 + p100
straddle = call_payoff(grid, Ks) + put_payoff(grid, Ks) - cost
out["straddle"] = {"strike": Ks, "call": c100, "put": p100, "cost": cost, "down": 80.0, "up": 120.0,
                   "profit_at_80": float(abs(80 - Ks) - cost), "profit_at_120": float(abs(120 - Ks) - cost),
                   "breakevens": [Ks - cost, Ks + cost], "profit": straddle.tolist(),
                   "worst": float(straddle.min())}
out["straddle"] | {"profit": "..."}

## 8. Check against QuantLib

The same two premiums from QuantLib's analytic Black-Scholes engine. The one-year option runs 29 September 2026 to 29 September 2027 on an actual/365 fixed year, so the time to expiry is exactly one year.

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365
ql_price = {}
for name, kind in (("call", ql.Option.Call), ("put", ql.Option.Put)):
    opt = ql.VanillaOption(ql.PlainVanillaPayoff(kind, K), ql.EuropeanExercise(expiry))
    opt.setPricingEngine(ql.AnalyticEuropeanEngine(process))
    ql_price[name] = opt.NPV()
out["quantlib"] = {"call": ql_price["call"], "put": ql_price["put"], "years": dc.yearFraction(today, expiry),
                   "max_diff": max(abs(ql_price["call"] - call_premium), abs(ql_price["put"] - put_premium))}
assert out["quantlib"]["max_diff"] < 1e-10
out["quantlib"]

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")